# 第 1 周末练习 —— 用本地 Ollama 做技术问答与简易宣传册

## 练习目标（理念）

为了展示你对 **OpenAI 兼容 API**（此处通过本地 **Ollama**）的熟悉程度，本笔记本做了几件小事：

1. 封装一个 `LLM_MODEL`：既能**一次性**拿完整回答，也能**流式**拿回答
2. 用几个「技术问题」任务练手（速度计算、环球旅行思路、生成爬虫代码）
3. 用生成的抓取/解析函数拉网页，再让模型根据页面内容**流式写出宣传册（brochure）**

## 和本课的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| OpenAI SDK + 自定义 `base_url` | `OpenAI(base_url='http://localhost:11434/v1/', ...)` |
| `messages`（system / user） | `sys_prompt` / `usr_prompt` |
| 非流式 vs 流式 | `ask_model` 与 `ask_model_stream` |
| 网页抓取 + 解析 | `requests` + BeautifulSoup |
| 流式 Markdown 刷新 | `display` + `update_display` |

## 怎么跑

1. 本机启动 Ollama，并确保已有 `llama3.2`（与常量 `MODEL_LLAMA` 一致）
2. 从上到下运行；任务 3 会打印模型生成的代码思路，任务 4 会真正请求网页并流式生成宣传册
3. 需要时可改各任务里的 `usr_prompt` 或任务 4 的 `webname` / `url`


In [ ]:
# ========== 导入：OpenAI 客户端 + 笔记本展示工具 ==========

# 从 openai 导入 OpenAI：后面会把它的 base_url 指到本地 Ollama 的 /v1 兼容接口
from openai import OpenAI
# display / Markdown：把模型回答渲染成 Markdown；update_display：流式时原地刷新同一块输出
from IPython.display import display, Markdown, update_display


In [ ]:
# ========== 常量：本练习实际使用的本地模型名 ==========

# 若要改回云端 GPT，可取消下一行注释并在别处接上对应客户端（当前主路径用 Llama）
# MODEL_GPT = 'gpt-4o-mini'
# 本地 Ollama 模型名：须与 `ollama list` 里已安装的名字一致
MODEL_LLAMA = 'llama3.2'


In [ ]:
# ========== 环境封装：一个类里提供「整段回答」和「流式回答」两种问法 ==========

class LLM_MODEL:

    def ask_model(self, sys_prompt, usr_prompt):
        # Ollama 的 OpenAI 兼容基址（注意末尾 /v1/）；不要改成别的路径除非你清楚端点差异
        model_url =  'http://localhost:11434/v1/'
        # 创建客户端：api_key 对本地 Ollama 多为占位，但 SDK 仍要求传入某个字符串
        client = OpenAI(base_url=model_url, api_key='ollama')
        # 组装 Chat Completions 的 messages：system 定人设，user 放具体问题
        msg = [{'role':'system', 'content':sys_prompt},{'role':'user', 'content':usr_prompt}]
        # 非流式调用：等模型整段生成完再返回
        response = client.chat.completions.create(model=MODEL_LLAMA, messages=msg)
        # 取出助手消息正文（content）返回给调用方
        return response.choices[0].message.content

    def ask_model_stream(self, sys_prompt, usr_prompt):
        # 与上面相同的本地端点与客户端创建方式
        model_url =  'http://localhost:11434/v1/'
        client = OpenAI(base_url=model_url, api_key='ollama')
        # 同样的 system / user 消息结构
        msg = [{'role':'system', 'content':sys_prompt},{'role':'user', 'content':usr_prompt}]
        # stream=True：返回可迭代的流对象，调用方再 for 循环逐块取 delta
        stream = client.chat.completions.create(model=MODEL_LLAMA, messages=msg, stream=True)
        return stream

# 实例化一次，后面多个任务单元格复用同一个 model 对象
model = LLM_MODEL()


In [ ]:
# ========== 任务 1：极速 —— 用非流式问答练手 ==========

# system：简短人设（保留英文 prompt，改译会改变回答风格）
sys_prompt = 'You are a helpful assistant who helps me understand technical questions.\n'
# user：一个速度相关的技术理解题（保留英文）
usr_prompt = 'It takes Alex 2 hours to travel a distance of 3 kms. What is the speed of Alex?'

# 调用非流式接口，拿到完整字符串回答
resp = model.ask_model(sys_prompt, usr_prompt)
# 在笔记本里用 Markdown 漂亮展示回答
display(Markdown(resp))


In [ ]:
# ========== 任务 2：最少天数环游世界？—— 开放性技术/规划题 ==========

# 同样的助手人设（可按任务改，但本格保持原样）
sys_prompt = 'You are a helpful assistant who helps me understand technical questions.\n'
# 开放性问题：考察模型组织长回答的能力（保留英文）
usr_prompt = 'There are many cities in our world. Can you tell me how to travel the whole world in least number of days ?'

# 非流式拿完整回答并 Markdown 展示
resp = model.ask_model(sys_prompt, usr_prompt)
display(Markdown(resp))


In [ ]:
# ========== 任务 3：请模型生成「抓网页 + 解析链接」的 Python 代码 ==========

# system：把模型定位成写代码的专家（保留英文）
sys_prompt = 'You are a coding expert who generates python code for given problem.\n'
# user：要求生成两个函数——取网页内容、解析页面中的链接（保留英文）
usr_prompt = 'Given a website URL, I want to a python function to get the contents of the webpage, and another function to parse all links in the given webpage text.'

# 非流式拿到模型生成的代码/说明文本
resp = model.ask_model(sys_prompt, usr_prompt)
# 这里用 print：方便原样查看代码块（不一定走 Markdown 渲染）
print(resp)


In [ ]:
# ========== 网页工具：抓取 HTML + 解析链接（供任务 4 使用） ==========

# 导入 requests：发 HTTP GET 拉网页正文
import requests
# 导入 BeautifulSoup：把 HTML 解析成可查询的树，方便找 <a href>
from bs4 import BeautifulSoup

def get_webpage_content(url):
    """
    Fetches the contents of a website.
    
    Args:
        url (str): URL of the webpage.
    
    Returns:
        str: HTML content of the webpage.
    """
    try:
        # 对目标 URL 发 GET 请求
        response = requests.get(url)
        # 若状态码是 4xx/5xx，抛出异常，进入下面的 except
        response.raise_for_status()  # Raise an exception for HTTP errors
        # 成功则返回响应文本（HTML 字符串）
        return response.text
    except requests.exceptions.RequestException as e:
        # 网络错误、超时、HTTP 错误等：打印后返回 None，让调用方自行判断
        print(f"Error fetching webpage: {e}")
        return None

def parse_links(html_content, base_url=""):
    """
    Parses links from a given HTML content.
    
    Args:
        html_content (str): HTML content of the webpage.
        base_url (str): Base URL to construct relative link URLs. Defaults to "".
    
    Returns:
        list: List of extracted URLs.
    """
    # 用 html.parser 解析整段 HTML
    soup = BeautifulSoup(html_content, 'html.parser')
    # 收集最终要返回的链接列表
    links = []

    # 遍历所有 <a> 锚点标签
    for tag in soup.find_all('a'):
        # 取出 href 属性；没有则为 None
        href = tag.get('href')

        # 处理绝对和相对 URL：无 href 或以 '/' 开头时，原逻辑置空字符串（相对链接未拼 base）
        if not href or href.startswith('/'):
            url = ""
        else:
            # 注意：条件写成 `if 0 and base_url`，恒为假，因此实际上总是走 else，直接用原始 href
            if 0 and base_url:
                url = f"{base_url}{href}"
            else:
                url = href

        # 只保留看起来像 https 的链接（原作者过滤条件；注意 'https:/' 也能匹配 'https://'）
        if url.startswith('https:/'):
            links.append(url)

    return links


In [ ]:
# ========== 任务 4：抓取站点内容，流式生成公司宣传册（brochure） ==========

# 用法示例：站点展示名 + 入口 URL（可改成你想分析的网站）
webname, url = 'Huggingface', "http://www.huggingface.co"

# 先抓首页 HTML
html_content = get_webpage_content(url)
# 从首页解析出一批 https 链接
links = parse_links(html_content, url)

print("Extracted Links:")
# 先把首页内容放进给模型的大字符串（后面会继续追加各 link）
content = f'Link:{url} -> Content:{html_content}\n'
# 遍历解析到的链接：打印出来，并（按原逻辑）再次请求——注意此处 get 的仍是入口 url，保持作者原代码不改
for link in links:
    print(link)
    html_content = get_webpage_content(url)
    content += f'Link:{link} -> Content:{html_content}\n'

# system：宣传册撰写助手（保留英文）
sys_prompt = 'You are a helpful assistant who helps me create a brochure for a website.\n'
# user：把站点名与抓到的内容拼进提示，请模型生成 brochure（保留英文模板与拼接逻辑）
usr_prompt = f'You are given the contents for a few pages for the website of {webname} following next line.\n' + \
             content + \
             'Use this information to give the brochure for this company.\n'

# 流式调用：返回 chunk 迭代器
stream = model.ask_model_stream(sys_prompt, usr_prompt)

# 累加完整回答；先放空 Markdown 占位以便原地刷新
response = ''
display_handle = display(Markdown(""), display_id=True)

# 逐块取 delta.content，拼起来并 update_display，形成打字机效果
for chunk in stream:
    response += chunk.choices[0].delta.content or ''
    update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# （空单元格）可留给你继续实验：例如换 URL、改 prompt，或把任务 3 生成的代码粘贴改造
